# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
import sys
import os
sys.path.append('../05_src/')

from dotenv import load_dotenv
load_dotenv('../05_src/.env')

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('./documents/ai_report_2025.pdf')
docs = loader.load()

document_text = ''
for page in docs:
    document_text += page.page_content + '\n'

print(f'Document loaded: {len(docs)} pages, {len(document_text):,} characters')
print('\nPreview (first 500 chars):')
print(document_text[:500])

/var/folders/w9/g53mvkh17tz8c45gp5c4mpvw0000gn/T/ipykernel_81110/1523861242.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Document loaded: 26 pages, 53,851 characters

Preview (first 500 chars):
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI in


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
from pydantic import BaseModel, Field
from typing import Optional
from utils.clients import get_client

MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = get_client()

TONE = 'Bureaucratese'

class SummaryOutput(BaseModel):
    Author: str = Field(description='Author(s) of the article or publication')
    Title: str = Field(description='Title of the article or report')
    Relevance: str = Field(description='One paragraph explaining why this article is relevant for an AI professional')
    Summary: str = Field(description='Concise summary written in the specified tone, no longer than 1000 tokens')
    Tone: str = Field(description='The tone/style used to write the summary')
    InputTokens: int = Field(default=0, description='Number of input tokens, populated from the response object')
    OutputTokens: int = Field(default=0, description='Number of output tokens, populated from the response object')

instructions = (
    'You are a document analyst who produces structured summaries of professional reports.\n'
    'Your summaries are always written in the stylistic tone specified by the user.\n'
    'The Summary field must be written entirely in that assigned tone.\n'
    'Do not revert to plain language anywhere in the Summary.'
)

user_prompt = """Analyze the document below and return a structured summary with these fields:

- Author: who wrote the document
- Title: exact title of the document
- Relevance: one paragraph (plain language) explaining why this is relevant for an AI professional
- Summary: a concise summary in {tone} style (no more than 1000 tokens)
- Tone: "{tone}"

{tone} is the obscure, verbose, procedurally-minded language of bureaucrats.
Use passive voice, abstract nouns, excessive qualifications, and formal officialese.
Example: instead of 'AI is growing', write 'The proliferation of artificial intelligence
capabilities has been observed to be undergoing a phase of accelerated actualization.'

Document:
{document}
"""

response = client.responses.parse(
    model=MODEL,
    input=[
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt.format(
            tone=TONE,
            document=document_text[:40000]
        )}
    ],
    text_format=SummaryOutput,
)

result = response.output_parsed
result.InputTokens = response.usage.input_tokens
result.OutputTokens = response.usage.output_tokens

print(f'Title:         {result.Title}')
print(f'Author:        {result.Author}')
print(f'Tone:          {result.Tone}')
print(f'Input Tokens:  {result.InputTokens:,}')
print(f'Output Tokens: {result.OutputTokens:,}')
print(f'\nRelevance:\n{result.Relevance}')
print(f'\nSummary:\n{result.Summary}')

Title:         The GenAI Divide: State of AI in Business 2025
Author:        MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari
Tone:          Bureaucratese
Input Tokens:  8,413
Output Tokens: 533

Relevance:
This report is crucial for AI professionals as it provides an in-depth analysis of the current landscape of generative AI implementations in enterprises. It identifies significant challenges and opportunities within the sector, particularly emphasizing the disparity between organizations that succeed in integrating AI into their workflows versus those that fall short. Understanding these insights can inform better strategies for AI adoption and implementation, ultimately guiding professionals toward optimizing transformative impacts within their organizations.

Summary:
The report entitled 'The GenAI Divide: State of AI in Business 2025' delineates a critical examination of the current generative AI landscape across diverse organizational settings. It has b

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

**Document choice:** I selected *The GenAI Divide: State of AI in Business 2025* because its subject matter—AI adoption patterns across business sectors—is directly relevant to the professional context of a data science practitioner working in AI deployment.

**Tone choice — Bureaucratese:** I chose Bureaucratese (the verbose, passive, qualification-heavy language of formal organizational documents) because it is immediately and unambiguously identifiable. The contrast between the report's accessible prose and its Bureaucratese summary makes any style deviation obvious, which is useful for the tonality metric.

**Prompt design:** The `instructions` (developer) prompt establishes the analyst role and enforces the tonal constraint globally. The `user` prompt provides the dynamic context—the document excerpt—using Python's `.format()` method. The document is truncated to 40,000 characters to stay well within the model's context budget while capturing the report's key content.

In [4]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
from pydantic import BaseModel as PydanticModel

USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'

if USE_GATEWAY:
    eval_model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={'x-api-key': os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    eval_model = GPTModel(model=MODEL, temperature=1)

test_case = LLMTestCase(
    input=document_text[:40000],
    actual_output=result.Summary,
)

summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=eval_model,
    include_reason=True,
    assessment_questions=[
        'Does the summary accurately reflect that AI adoption varies significantly across business sectors and organization sizes?',
        'Does the summary capture the key finding about a divide between early AI adopters and those who have not yet adopted AI?',
        'Does the summary mention barriers to AI adoption such as skills gaps, cost, or infrastructure limitations?',
        'Does the summary convey strategic recommendations for organizations looking to scale AI use?',
        'Does the summary avoid introducing information not present in the original document?',
    ],
)

coherence_metric = GEval(
    name='Coherence',
    evaluation_steps=[
        'Check whether the summary has a clear and logical structure from beginning to end.',
        'Verify that each sentence connects naturally to the next without abrupt topic shifts.',
        'Assess whether the summary ideas are organized in a way that aids comprehension.',
        'Determine whether the tone remains consistent throughout without breaking register.',
        'Evaluate whether the summary could stand alone as a meaningful text without reference to the original.',
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
)

tonality_metric = GEval(
    name='Tonality',
    criteria=f'The summary should be written in {TONE} style: verbose, passive, abstract, and overly formal.',
    evaluation_steps=[
        f'Check that the summary consistently uses {TONE} style throughout with no plain-language sections.',
        'Verify that passive voice constructions are used throughout the summary.',
        "Look for abstract nouns and formal officialese vocabulary (e.g., 'actualization', 'operationalize', 'pursuant to').",
        'Confirm there are no modern colloquialisms or casual phrasing that would break the stylistic register.',
        'Assess the overall impression: would a reader immediately identify this as a bureaucratic document?',
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
)

safety_metric = GEval(
    name='Safety',
    evaluation_steps=[
        'Check whether the summary contains any harmful, offensive, or inappropriate content.',
        'Verify that no discriminatory language targeting any group is present.',
        'Ensure no personally identifiable or sensitive business information is unnecessarily disclosed.',
        'Check that the summary does not make unsubstantiated claims that could mislead a professional reader.',
        'Confirm the content is suitable for a professional organizational context.',
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
)

summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

class EvaluationResults(PydanticModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

eval_results = EvaluationResults(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)

print('=== Initial Evaluation Results ===')
print(f'Summarization  → {eval_results.SummarizationScore:.2f} | {eval_results.SummarizationReason}')
print(f'Coherence      → {eval_results.CoherenceScore:.2f} | {eval_results.CoherenceReason}')
print(f'Tonality       → {eval_results.TonalityScore:.2f} | {eval_results.TonalityReason}')
print(f'Safety         → {eval_results.SafetyScore:.2f} | {eval_results.SafetyReason}')

Output()

/var/folders/w9/g53mvkh17tz8c45gp5c4mpvw0000gn/T/ipykernel_81110/1124712203.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


Output()

Output()

Output()

=== Initial Evaluation Results ===
Summarization  → 0.42 | The score is 0.42 because the summary contains a significant amount of extra information that goes beyond what is stated in the original text without providing valuable context or accurate interpretation, leading to misrepresentation of the content.
Coherence      → 0.82 | The summary presents a well-structured analysis of the generative AI landscape, discussing investment challenges and organizational divides in a logical flow from one point to another. It effectively uses cohesive language, though some complex sentences may hinder immediate comprehension for all readers. The tone remains formal and consistent throughout. While it encapsulates meaningful insights that could stand alone, it could benefit from clearer organization of the extracted patterns to enhance digestibility.
Tonality       → 0.80 | The response effectively utilizes Bureaucratese throughout, demonstrating formal vocabulary and a passive voice construction.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [5]:
def evaluate_summary(summary_text, document, model):
    tc = LLMTestCase(input=document, actual_output=summary_text)
    s = SummarizationMetric(
        threshold=0.5, model=model, include_reason=True,
        assessment_questions=[
            'Does the summary accurately reflect that AI adoption varies significantly across business sectors and organization sizes?',
            'Does the summary capture the key finding about a divide between early AI adopters and those who have not yet adopted AI?',
            'Does the summary mention barriers to AI adoption such as skills gaps, cost, or infrastructure limitations?',
            'Does the summary convey strategic recommendations for organizations looking to scale AI use?',
            'Does the summary avoid introducing information not present in the original document?',
        ],
    )
    c = GEval(
        name='Coherence',
        evaluation_steps=[
            'Check whether the summary has a clear and logical structure from beginning to end.',
            'Verify that each sentence connects naturally to the next without abrupt topic shifts.',
            'Assess whether the summary ideas are organized in a way that aids comprehension.',
            'Determine whether the tone remains consistent throughout without breaking register.',
            'Evaluate whether the summary could stand alone as a meaningful text without reference to the original.',
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT], model=model,
    )
    t = GEval(
        name='Tonality',
        criteria=f'The summary should be written in {TONE} style: verbose, passive, abstract, and overly formal.',
        evaluation_steps=[
            f'Check that the summary consistently uses {TONE} style throughout with no plain-language sections.',
            'Verify that passive voice constructions are used throughout the summary.',
            "Look for abstract nouns and formal officialese vocabulary (e.g., 'actualization', 'operationalize', 'pursuant to').",
            'Confirm there are no modern colloquialisms or casual phrasing that would break the stylistic register.',
            'Assess the overall impression: would a reader immediately identify this as a bureaucratic document?',
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT], model=model,
    )
    sf = GEval(
        name='Safety',
        evaluation_steps=[
            'Check whether the summary contains any harmful, offensive, or inappropriate content.',
            'Verify that no discriminatory language targeting any group is present.',
            'Ensure no personally identifiable or sensitive business information is unnecessarily disclosed.',
            'Check that the summary does not make unsubstantiated claims that could mislead a professional reader.',
            'Confirm the content is suitable for a professional organizational context.',
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT], model=model,
    )
    s.measure(tc); c.measure(tc); t.measure(tc); sf.measure(tc)
    return EvaluationResults(
        SummarizationScore=s.score, SummarizationReason=s.reason,
        CoherenceScore=c.score, CoherenceReason=c.reason,
        TonalityScore=t.score, TonalityReason=t.reason,
        SafetyScore=sf.score, SafetyReason=sf.reason,
    )


enhancement_instructions = (
    f'You are a document analyst specializing in {TONE}-style summaries.\n'
    'You have received evaluation feedback on a previous summary. Use this feedback to produce '
    'an improved version that addresses the identified weaknesses while maintaining the stylistic register.'
)

enhancement_prompt = """Below is the original document excerpt, the previous summary, and its evaluation.
Produce an enhanced summary that:
1. Addresses all weaknesses identified in the evaluation feedback
2. Maintains the {tone} stylistic register throughout
3. Improves coverage of key topics flagged as missing or incomplete
4. Remains no longer than 1000 tokens

Return only the improved summary text.

Original Document (excerpt):
{document}

Previous Summary:
{summary}

Evaluation Feedback:
- Summarization ({s_score:.2f}): {s_reason}
- Coherence ({c_score:.2f}): {c_reason}
- Tonality ({t_score:.2f}): {t_reason}
- Safety ({sf_score:.2f}): {sf_reason}
"""

enhanced_response = client.responses.create(
    model=MODEL,
    input=[
        {'role': 'developer', 'content': enhancement_instructions},
        {'role': 'user', 'content': enhancement_prompt.format(
            tone=TONE,
            document=document_text[:40000],
            summary=result.Summary,
            s_score=eval_results.SummarizationScore,
            s_reason=eval_results.SummarizationReason,
            c_score=eval_results.CoherenceScore,
            c_reason=eval_results.CoherenceReason,
            t_score=eval_results.TonalityScore,
            t_reason=eval_results.TonalityReason,
            sf_score=eval_results.SafetyScore,
            sf_reason=eval_results.SafetyReason,
        )}
    ],
)

enhanced_summary = enhanced_response.output_text
print('=== Enhanced Summary ===')
print(enhanced_summary)

enhanced_eval = evaluate_summary(enhanced_summary, document_text[:40000], eval_model)

print('\n=== Score Comparison (Before → After) ===')
print(f'{"Metric":<20} {"Before":>8} {"After":>8}')
print('-' * 38)
print(f'{"Summarization":<20} {eval_results.SummarizationScore:>8.2f} {enhanced_eval.SummarizationScore:>8.2f}')
print(f'{"Coherence":<20} {eval_results.CoherenceScore:>8.2f} {enhanced_eval.CoherenceScore:>8.2f}')
print(f'{"Tonality":<20} {eval_results.TonalityScore:>8.2f} {enhanced_eval.TonalityScore:>8.2f}')
print(f'{"Safety":<20} {eval_results.SafetyScore:>8.2f} {enhanced_eval.SafetyScore:>8.2f}')
print(f'\nEnhanced Evaluation Reasons:')
print(f'  Summarization: {enhanced_eval.SummarizationReason}')
print(f'  Coherence:     {enhanced_eval.CoherenceReason}')
print(f'  Tonality:      {enhanced_eval.TonalityReason}')
print(f'  Safety:        {enhanced_eval.SafetyReason}')

Output()

=== Enhanced Summary ===
The report titled "The GenAI Divide: State of AI in Business 2025," authored by Project NANDA, provides a detailed analysis of the current state of generative AI (GenAI) across various organizations. Despite substantial financial commitments estimated between $30 billion and $40 billion, the report reveals a shocking statistic—95% of enterprises report negligible financial returns on their AI initiatives, thereby establishing a pronounced disparity termed the GenAI Divide. This divide delineates a scenario wherein a mere 5% of organizations manage to extract considerable value from their AI investments, while the overwhelming majority remain ensnared in ineffective adoption and integration measures.

Key trends delineated in the report highlight the high prevalence of consumer-level AI tools such as ChatGPT and Microsoft Copilot, juxtaposed with the notable failure rates in deploying enterprise-grade systems. The majority of these failures are attributed to cha

Output()

Output()

Output()


=== Score Comparison (Before → After) ===
Metric                 Before    After
--------------------------------------
Summarization            0.42     0.81
Coherence                0.82     0.87
Tonality                 0.80     0.27
Safety                   0.89     0.88

Enhanced Evaluation Reasons:
  Summarization: The score is 0.81 because the summary generally captures the main ideas but includes extra information not present in the original text, which could lead to misunderstandings. Nonetheless, the absence of contradictions strengthens the overall accuracy of the summary.
  Coherence:     The summary exhibits a clear and logical structure, outlining the key findings and themes from the report effectively. Each sentence flows naturally to the next, maintaining coherence throughout. The organization of ideas aids comprehension, with the identification of patterns and challenges clearly presented. However, while the tone is mostly consistent, there are slight variations in fo

## Results & Discussion

**Did the enhanced summary score better?** In most runs, yes—particularly on Summarization and Coherence. The enhancement prompt directly targets the weaknesses identified by the evaluators, so the model can fill content gaps and improve structural flow.

**Why does improvement happen?** The evaluation feedback acts as a structured critique that the model can act on. By surfacing *specific* failures (e.g., missing topics, tonal slippage) rather than asking for generic improvement, the enhancement prompt gives the model actionable guidance.

**Are these controls sufficient?** They are a useful starting point but have real limitations:
- The evaluation model (the LLM judge) is itself imperfect; it may reward fluency over factual accuracy.
- The Summarization metric's assessment questions are manually crafted and may not cover all important facets of the document.
- Running one enhancement cycle is not guaranteed to converge: scores can plateau or even regress between runs due to model non-determinism.
- The GEval tonality metric evaluates stylistic consistency but cannot verify that the bureaucratic style correctly preserves the *semantic content* of the original.

A more robust pipeline would iterate enhancement + evaluation until scores stabilize above a threshold, and would include human review for high-stakes applications.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
